## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 100], stddev=0.1), name='W1')  # 输入层到隐藏层权重
        self.b1 = tf.Variable(tf.zeros([100]), name='b1')  # 隐藏层偏置
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1), name='W2')  # 隐藏层到输出层权重
        self.b2 = tf.Variable(tf.zeros([10]), name='b2')  # 输出层偏置
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 28 * 28])  # 将图像拉平成向量
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)  # 第一层全连接并接 ReLU
        logits = tf.matmul(h1, self.W2) + self.b2  # 输出未归一化的 logits
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.5193043 ; accuracy 0.10966667
epoch 1 : loss 2.484237 ; accuracy 0.1136
epoch 2 : loss 2.4541626 ; accuracy 0.11771667
epoch 3 : loss 2.4279528 ; accuracy 0.1222
epoch 4 : loss 2.404771 ; accuracy 0.12618333
epoch 5 : loss 2.383999 ; accuracy 0.13048333
epoch 6 : loss 2.3651636 ; accuracy 0.13581666
epoch 7 : loss 2.3479083 ; accuracy 0.1408
epoch 8 : loss 2.3319561 ; accuracy 0.14605
epoch 9 : loss 2.317091 ; accuracy 0.15131667
epoch 10 : loss 2.303138 ; accuracy 0.15705
epoch 11 : loss 2.289962 ; accuracy 0.16238333
epoch 12 : loss 2.2774518 ; accuracy 0.1684
epoch 13 : loss 2.2655182 ; accuracy 0.17465
epoch 14 : loss 2.2540874 ; accuracy 0.17965
epoch 15 : loss 2.2430995 ; accuracy 0.18526667
epoch 16 : loss 2.2325037 ; accuracy 0.1915
epoch 17 : loss 2.2222567 ; accuracy 0.19735
epoch 18 : loss 2.2123213 ; accuracy 0.2027
epoch 19 : loss 2.2026677 ; accuracy 0.2087
epoch 20 : loss 2.1932685 ; accuracy 0.21496667
epoch 21 : loss 2.1840992 ; accuracy 0.22045
epoch 